# One-head nanoGPT optimizer comparison

This notebook compares the completed **SGD + Nesterov**, **AdamW**, and **Muon + auxiliary AdamW** baselines under one matched one-block/one-head FineWeb-Edu protocol.

All epoch trajectories show individual runs, the across-seed mean, and a two-sided **95% Student-t** confidence interval. It compares `train_accuracy`, `test_accuracy`, `train_loss`, `test_loss`, perplexity, fixed-continuation BLEU, layer `alpha`, `ERG_gap`, and direct `num_traps`.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

cwd = Path.cwd().resolve()
candidates = [cwd, cwd.parent, cwd / "baseline" / "nanogpt_one_head"]
EXPERIMENT_ROOT = next(
    (path for path in candidates if (path / "configs" / "reference.yaml").is_file()),
    None,
)
if EXPERIMENT_ROOT is None:
    raise FileNotFoundError("Run from baseline/nanogpt_one_head or the repository root")
sys.path.insert(0, str(EXPERIMENT_ROOT / "src"))

from rg_nanogpt_one_head import (
    MATRIX_COLORS,
    OPTIMIZER_COLORS,
    SUPPORTED_OPTIMIZERS,
    canonical_seeds,
    final_test_summary,
    load_config,
    load_epoch_metrics,
    load_layer_metrics,
    load_spectral_summary,
    load_test_results,
    plot_epoch_metric,
    plot_layer_metric,
    plot_spectral_optimizer_summary,
    roots,
    run_status_table,
)

CONFIG = load_config(EXPERIMENT_ROOT / "configs" / "reference.yaml")
SEEDS = canonical_seeds(CONFIG)
OPTIMIZERS = SUPPORTED_OPTIMIZERS
PATHS = roots()
PATHS["plots"].mkdir(parents=True, exist_ok=True)
display(run_status_table(PATHS["results"], optimizers=OPTIMIZERS, seeds=SEEDS))
display(pd.DataFrame({"optimizer": list(OPTIMIZER_COLORS), "color": list(OPTIMIZER_COLORS.values())}))
display(pd.DataFrame({"matrix_type": list(MATRIX_COLORS), "color": list(MATRIX_COLORS.values())}))

## Load the complete nine-run suite

This fails if any optimizer/seed run is incomplete. The test set stayed held out until post-training audits of the final and validation-selected checkpoints; it was never used for updates, schedule selection, early stopping, or hyperparameter tuning.

In [ ]:
epoch_metrics = load_epoch_metrics(PATHS["results"], optimizers=OPTIMIZERS, seeds=SEEDS)
layer_metrics = load_layer_metrics(PATHS["results"], optimizers=OPTIMIZERS, seeds=SEEDS)
spectral_summary = load_spectral_summary(PATHS["results"], optimizers=OPTIMIZERS, seeds=SEEDS)
test_results = load_test_results(PATHS["results"], optimizers=OPTIMIZERS, seeds=SEEDS)
display(epoch_metrics.sort_values(["optimizer", "seed", "nominal_epoch"]))

## Task metrics by epoch

In [ ]:
comparison_plot_dir = PATHS["plots"] / "comparison"
for metric in [
    "train_loss", "val_loss", "test_loss",
    "train_accuracy", "val_accuracy", "test_accuracy",
    "train_perplexity", "val_perplexity", "test_perplexity",
    "test_bleu", "val_generalization_gap", "test_generalization_gap",
]:
    plot_epoch_metric(
        epoch_metrics,
        metric=metric,
        optimizers=OPTIMIZERS,
        title=f"One-head nanoGPT: {metric} (95% Student-t CI)",
        output=comparison_plot_dir / f"{metric}.png",
    )
    plt.show()

## Optimizer-level WeightWatcher summaries

In [ ]:
for metric in ["alpha_median", "ERG_gap_median", "num_traps_mean"]:
    plot_spectral_optimizer_summary(
        spectral_summary,
        metric=metric,
        optimizers=OPTIMIZERS,
        output=comparison_plot_dir / f"spectral_{metric}.png",
    )
    if metric == "alpha_median":
        plt.axhline(2.0, color="black", linestyle="--", linewidth=1.0)
    if metric == "ERG_gap_median":
        plt.axhline(0.0, color="black", linestyle="--", linewidth=1.0)
    plt.show()

## Layer-resolved alpha, ERG gap, and correlation traps

The matrix color map is invariant across optimizers, so the same spectral object is visually comparable in every panel.

In [ ]:
for optimizer in OPTIMIZERS:
    for metric in ["alpha", "ERG_gap", "num_traps"]:
        plot_layer_metric(
            layer_metrics,
            optimizer=optimizer,
            metric=metric,
            title=f"{optimizer}: layer {metric} (95% Student-t CI)",
            output=comparison_plot_dir / f"{optimizer}_layer_{metric}.png",
        )
        plt.show()

## Final and validation-selected test summaries

These tables report run-level 95% Student-t intervals across the three independent seeds.

In [ ]:
summary = final_test_summary(test_results)
display(
    summary[
        [
            "optimizer_label", "checkpoint", "metric", "n", "mean", "sd",
            "ci95_half_width", "ci95_lower", "ci95_upper",
        ]
    ].sort_values(["checkpoint", "metric", "optimizer_label"])
)
summary.to_csv(comparison_plot_dir / "final_test_summary_95ci.csv", index=False)